## 1. Purpose

This notebook transforms the cleaned session-level dataset into participant-level trajectory features. Rather than analysing individual therapy sessions, multiple sessions belonging to each participant are summarised into numerical features that describe behavioural patterns and progress over time. These features will be used for clustering and predictive modelling in later stages of the dissertation.


## 7. Validate new dataset

## 8. Save participant feature dataset

## 9. Summary

## 2. Import libraries

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

## 3. Set project paths

In [2]:
project_root = Path.cwd().parent

processed_data_folder = project_root / "data" / "processed"

output_folder = project_root / "outputs"

## 4. Load cleaned dataset

In [3]:
df = pd.read_csv(
    processed_data_folder / "cleaned_data.csv"
)

In [4]:
df.head()

,Participant id,Age,Gender,Special_interest,Any co-existing disabiltiy diagnosis,Co-existing disability diagnosed details,Level of Severity,Autism Level,Stimming behaviour Identified?,Observed_stimming,...,How much different scenarios stories impact overall social behaviour ?,Comment_Q26,Theme_specific_situation,Engagement_opportunities_count,Successful/postive response,Success_percentage,Notes_intervention,Additional_notes_observations,response_time_seconds,success_percentage_calculated
0,101,10,Male,"Shopping at the supermarket, going to the gas ...",Yes,"He has ADHD,which is currently being treated w...",6,2,Yes,Repetive questioning and repetive speecch,...,3,NaN,No Translation,10.0,3.0,30%,NaN,The participant demonstrated very good underst...,NaN,30.000000
1,101,10,Male,"Shopping at the supermarket, going to the gas ...",Yes,"He has ADHD,which is currently being treated w...",6,2,Yes,Repetive questioning and repetive speecch,...,4,NaN,No Translation,10.0,5.0,50%,NaN,The participant showed enthusiasm for listenin...,NaN,50.000000
2,101,10,Male,"Shopping at the supermarket, going to the gas ...",Yes,"He has ADHD,which is currently being treated w...",6,2,Yes,Repetive questioning and repetive speecch,...,5,NaN,No Translation,10.0,5.0,50%,NaN,The participant continued to retain elements o...,NaN,50.000000
3,101,10,Male,"Shopping at the supermarket, going to the gas ...",Yes,"He has ADHD,which is currently being treated w...",6,2,Yes,Repetive questioning and repetive speecch,...,5,NaN,No Translation,3.0,1.0,30%,NaN,"At the beginning of the class, an appropriate ...",NaN,33.333333
4,101,10,Male,"Shopping at the supermarket, going to the gas ...",Yes,"He has ADHD,which is currently being treated w...",6,2,Yes,Repetive questioning and repetive speecch,...,7,NaN,No Translation,8.0,4.0,50%,NaN,"According to the family, the stories have been...",NaN,50.000000


In [5]:
df.shape

(512, 75)

In [6]:
df["Participant id"].nunique()

32

## 5. Explore Participant Sessions

The cleaned dataset contains multiple therapy sessions for each participant. Before creating participant-level features, the data are grouped by participant identifier. This enables summary statistics to be calculated across all sessions belonging to the same participant.

In [7]:
participant_groups = df.groupby("Participant id")

In [8]:
participant_groups

## 6. Create Participant Trajectory Features

The first trajectory feature represents the number of storytelling sessions completed by each participant. This provides an indication of therapy exposure and participant engagement over the study period.

In [11]:
participant_features = pd.DataFrame()

participant_features["Number_of_sessions"] = (
    df.groupby("Participant id")["Session number"].nunique()
)

In [12]:
participant_features.head()

,Number_of_sessions
Participant id,
101,8
102,8
104,8
105,8
106,8


### Number of Sessions

Each participant had eight intervention sessions. However, the dataset contained separate parent and therapist assessments for each session, resulting in 16 records per participant. Therefore, the number of intervention sessions was calculated using the number of unique session identifiers rather than the total number of records.

### Mean Engagement Score

The mean engagement score was calculated for each participant by averaging engagement ratings across all intervention sessions. This feature represents the participant's overall level of engagement throughout the storytelling intervention.

In [13]:
df.columns[df.columns.str.contains("engaged", case=False)]

Index(['How engaged was the participant during today's storytelling session?'], dtype='object')

In [14]:
engagement_col = "How engaged was the participant during today's storytelling session?"

In [15]:
participant_features["Mean_engagement"] = (
    df.groupby("Participant id")[engagement_col].mean()
)

In [16]:
participant_features.head()

,Number_of_sessions,Mean_engagement
Participant id,,
101,8,2.437500
102,8,1.750000
104,8,2.187500
105,8,3.562500
106,8,1.533333


### Mean Emotional Connection

The mean emotional connection score was calculated for each participant across all available intervention records. This feature summarises the extent to which the participant demonstrated emotional connection during the storytelling intervention.

In [17]:
emotional_col = "Did the participant demonstrate emotional connection?"

participant_features["Mean_emotional_connection"] = (
    df.groupby("Participant id")[emotional_col].mean()
)

In [18]:
participant_features.head()

,Number_of_sessions,Mean_engagement,Mean_emotional_connection
Participant id,,,
101,8,2.437500,2.125000
102,8,1.750000,1.687500
104,8,2.187500,1.687500
105,8,3.562500,2.875000
106,8,1.533333,1.666667


In [19]:
df.columns[df.columns.str.contains("verbal", case=False)]

Index(['How would you rate the participant’s verbal participation?', 'Did the participant express their feelings about the story (verbally or otherwise)?'], dtype='object')

In [20]:
verbal_col = "How would you rate the participant’s verbal participation?"

In [21]:
participant_features["Mean_verbal_participation"] = (
    df.groupby("Participant id")[verbal_col].mean()
)

In [22]:
participant_features.head()

,Number_of_sessions,Mean_engagement,Mean_emotional_connection,Mean_verbal_participation
Participant id,,,,
101,8,2.437500,2.125000,6.250000
102,8,1.750000,1.687500,4.000000
104,8,2.187500,1.687500,5.333333
105,8,3.562500,2.875000,7.625000
106,8,1.533333,1.666667,3.800000


### Selection of Mean Behavioural Features

Several quantitative variables were selected to represent different dimensions of participant engagement and behaviour. These variables will be averaged across the available intervention records for each participant to create participant-level summary features.

In [23]:
mean_feature_map = {
    "How engaged was the participant during today's storytelling session?":
        "Mean_engagement",

    "Did the participant demonstrate emotional connection?":
        "Mean_emotional_connection",

    "How would you rate the participant’s verbal participation?":
        "Mean_verbal_participation",

    "Did the participant maintain attention throughout the session?":
        "Mean_attention",

    "Did the participant show any signs of enjoyment during the session?":
        "Mean_enjoyment",

    "To what extent did the participant understand the story theme?":
        "Mean_story_understanding"
}

In [24]:
mean_features = (
    df.groupby("Participant id")[list(mean_feature_map.keys())]
    .mean()
    .rename(columns=mean_feature_map)
)

In [25]:
mean_features.head()

,Mean_engagement,Mean_emotional_connection,Mean_verbal_participation,Mean_attention,Mean_enjoyment,Mean_story_understanding
Participant id,,,,,,
101,2.437500,2.125000,6.250000,2.562500,2.812500,2.571429
102,1.750000,1.687500,4.000000,1.750000,2.250000,1.812500
104,2.187500,1.687500,5.333333,2.250000,2.266667,2.200000
105,3.562500,2.875000,7.625000,3.437500,3.187500,3.500000
106,1.533333,1.666667,3.800000,1.666667,1.500000,1.866667


In [28]:
participant_features = pd.DataFrame()

In [29]:
participant_features["Number_of_sessions"] = (
    df.groupby("Participant id")["Session number"].nunique()
)

In [30]:
participant_features = participant_features.join(mean_features)

In [31]:
participant_features.head()

,Number_of_sessions,Mean_engagement,Mean_emotional_connection,Mean_verbal_participation,Mean_attention,Mean_enjoyment,Mean_story_understanding
Participant id,,,,,,,
101,8,2.437500,2.125000,6.250000,2.562500,2.812500,2.571429
102,8,1.750000,1.687500,4.000000,1.750000,2.250000,1.812500
104,8,2.187500,1.687500,5.333333,2.250000,2.266667,2.200000
105,8,3.562500,2.875000,7.625000,3.437500,3.187500,3.500000
106,8,1.533333,1.666667,3.800000,1.666667,1.500000,1.866667


### Participant-Level Mean Features

The cleaned session-level dataset was aggregated by participant identifier to create one row per participant. The number of intervention sessions was calculated using the number of unique session identifiers, while selected behavioural variables were summarised using their mean across all available parent and therapist records. These features included engagement, emotional connection, verbal participation, attention, enjoyment, and story understanding. Missing ratings were excluded from the mean calculations.

In [32]:
participant_features["Mean_response_time_seconds"] = (
    df.groupby("Participant id")["response_time_seconds"].mean()
)

In [33]:
participant_features.head()

,Number_of_sessions,Mean_engagement,Mean_emotional_connection,Mean_verbal_participation,Mean_attention,Mean_enjoyment,Mean_story_understanding,Mean_response_time_seconds
Participant id,,,,,,,,
101,8,2.437500,2.125000,6.250000,2.562500,2.812500,2.571429,NaN
102,8,1.750000,1.687500,4.000000,1.750000,2.250000,1.812500,160.0000
104,8,2.187500,1.687500,5.333333,2.250000,2.266667,2.200000,6.2500
105,8,3.562500,2.875000,7.625000,3.437500,3.187500,3.500000,42.1875
106,8,1.533333,1.666667,3.800000,1.666667,1.500000,1.866667,77.5000


In [34]:
participant_features["Mean_success_percentage"] = (
    df.groupby("Participant id")["success_percentage_calculated"].mean()
)

In [35]:
participant_features.head()

,Number_of_sessions,Mean_engagement,Mean_emotional_connection,Mean_verbal_participation,Mean_attention,Mean_enjoyment,Mean_story_understanding,Mean_response_time_seconds,Mean_success_percentage
Participant id,,,,,,,,,
101,8,2.437500,2.125000,6.250000,2.562500,2.812500,2.571429,NaN,51.904762
102,8,1.750000,1.687500,4.000000,1.750000,2.250000,1.812500,160.0000,64.895833
104,8,2.187500,1.687500,5.333333,2.250000,2.266667,2.200000,6.2500,64.000000
105,8,3.562500,2.875000,7.625000,3.437500,3.187500,3.500000,42.1875,80.625000
106,8,1.533333,1.666667,3.800000,1.666667,1.500000,1.866667,77.5000,61.047619


### Mean Success Percentage

The mean success percentage was calculated for each participant using the cleaned and internally consistent `success_percentage_calculated` variable. This feature represents the average proportion of successful responses across the participant’s intervention records.

In [36]:
participant_features.columns

Index(['Number_of_sessions', 'Mean_engagement', 'Mean_emotional_connection',
       'Mean_verbal_participation', 'Mean_attention', 'Mean_enjoyment',
       'Mean_story_understanding', 'Mean_response_time_seconds',
       'Mean_success_percentage'],
      dtype='object')

In [37]:
mean_feature_map = {
    "How engaged was the participant during today's storytelling session?":
        "Mean_engagement",

    "Did the participant demonstrate emotional connection?":
        "Mean_emotional_connection",

    "How would you rate the participant’s verbal participation?":
        "Mean_verbal_participation",

    "Did the participant maintain attention throughout the session?":
        "Mean_attention",

    "Did the participant show any signs of enjoyment during the session?":
        "Mean_enjoyment",

    "To what extent did the participant understand the story theme?":
        "Mean_story_understanding",

    "Did the participant apply the learning during the session?":
        "Mean_learning_application",

    "Do you feel that the participant is confident and has the potential to be able to apply this story in the real world after a session?":
        "Mean_confidence",

    "Did the participant generalise the behaviour outside the story?":
        "Mean_generalisation",

    "Did the participant recall or refer to a previous story or theme?":
        "Mean_story_recall",

    "Did the participant reflect on or comment about the theme after the story ended?":
        "Mean_reflection",

    "Did the participant link the story to real-life experiences?":
        "Mean_real_life_connection"
}

In [38]:
mean_features = (
    df.groupby("Participant id")[list(mean_feature_map.keys())]
    .mean()
    .rename(columns=mean_feature_map)
)

In [39]:
mean_features.head()

,Mean_engagement,Mean_emotional_connection,Mean_verbal_participation,Mean_attention,Mean_enjoyment,Mean_story_understanding,Mean_learning_application,Mean_confidence,Mean_generalisation,Mean_story_recall,Mean_reflection,Mean_real_life_connection
Participant id,,,,,,,,,,,,
101,2.437500,2.125000,6.250000,2.562500,2.812500,2.571429,2.384615,2.000000,1.916667,2.083333,1.538462,1.461538
102,1.750000,1.687500,4.000000,1.750000,2.250000,1.812500,1.437500,0.800000,0.600000,0.750000,0.250000,0.375000
104,2.187500,1.687500,5.333333,2.250000,2.266667,2.200000,1.666667,1.285714,0.933333,1.285714,0.666667,1.066667
105,3.562500,2.875000,7.625000,3.437500,3.187500,3.500000,2.000000,2.375000,2.125000,2.312500,1.875000,2.187500
106,1.533333,1.666667,3.800000,1.666667,1.500000,1.866667,1.000000,1.000000,0.600000,0.933333,0.466667,0.666667


In [40]:
participant_features = pd.DataFrame()

participant_features["Number_of_sessions"] = (
    df.groupby("Participant id")["Session number"].nunique()
)

participant_features = participant_features.join(mean_features)

participant_features["Mean_response_time_seconds"] = (
    df.groupby("Participant id")["response_time_seconds"].mean()
)

participant_features["Mean_success_percentage"] = (
    df.groupby("Participant id")["success_percentage_calculated"].mean()
)

In [41]:
participant_features.head()

,Number_of_sessions,Mean_engagement,Mean_emotional_connection,Mean_verbal_participation,Mean_attention,Mean_enjoyment,Mean_story_understanding,Mean_learning_application,Mean_confidence,Mean_generalisation,Mean_story_recall,Mean_reflection,Mean_real_life_connection,Mean_response_time_seconds,Mean_success_percentage
Participant id,,,,,,,,,,,,,,,
101,8,2.437500,2.125000,6.250000,2.562500,2.812500,2.571429,2.384615,2.000000,1.916667,2.083333,1.538462,1.461538,NaN,51.904762
102,8,1.750000,1.687500,4.000000,1.750000,2.250000,1.812500,1.437500,0.800000,0.600000,0.750000,0.250000,0.375000,160.0000,64.895833
104,8,2.187500,1.687500,5.333333,2.250000,2.266667,2.200000,1.666667,1.285714,0.933333,1.285714,0.666667,1.066667,6.2500,64.000000
105,8,3.562500,2.875000,7.625000,3.437500,3.187500,3.500000,2.000000,2.375000,2.125000,2.312500,1.875000,2.187500,42.1875,80.625000
106,8,1.533333,1.666667,3.800000,1.666667,1.500000,1.866667,1.000000,1.000000,0.600000,0.933333,0.466667,0.666667,77.5000,61.047619


In [42]:
participant_features.shape

(32, 15)

In [43]:
participant_features.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32 entries, 101 to 136
Data columns (total 15 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Number_of_sessions          32 non-null     int64  
 1   Mean_engagement             32 non-null     float64
 2   Mean_emotional_connection   32 non-null     float64
 3   Mean_verbal_participation   32 non-null     float64
 4   Mean_attention              32 non-null     float64
 5   Mean_enjoyment              32 non-null     float64
 6   Mean_story_understanding    32 non-null     float64
 7   Mean_learning_application   32 non-null     float64
 8   Mean_confidence             32 non-null     float64
 9   Mean_generalisation         32 non-null     float64
 10  Mean_story_recall           32 non-null     float64
 11  Mean_reflection             32 non-null     float64
 12  Mean_real_life_connection   32 non-null     float64
 13  Mean_response_time_seconds  27 non-null

In [44]:
participant_features.isna().sum()

Number_of_sessions            0
Mean_engagement               0
Mean_emotional_connection     0
Mean_verbal_participation     0
Mean_attention                0
Mean_enjoyment                0
Mean_story_understanding      0
Mean_learning_application     0
Mean_confidence               0
Mean_generalisation           0
Mean_story_recall             0
Mean_reflection               0
Mean_real_life_connection     0
Mean_response_time_seconds    5
Mean_success_percentage       1
dtype: int64

In [45]:
participant_features.describe().round(2)

,Number_of_sessions,Mean_engagement,Mean_emotional_connection,Mean_verbal_participation,Mean_attention,Mean_enjoyment,Mean_story_understanding,Mean_learning_application,Mean_confidence,Mean_generalisation,Mean_story_recall,Mean_reflection,Mean_real_life_connection,Mean_response_time_seconds,Mean_success_percentage
count,32.0,32.00,32.00,32.00,32.00,32.00,32.00,32.00,32.00,32.00,32.00,32.00,32.00,27.00,31.00
mean,8.0,2.86,2.62,7.03,2.82,2.77,2.91,2.08,2.18,1.87,2.34,1.94,1.89,69.82,67.91
std,0.0,0.83,0.79,2.37,0.81,0.78,0.86,0.84,0.91,0.87,1.02,0.97,0.87,48.67,22.89
min,8.0,0.40,0.10,0.40,0.30,0.30,0.00,0.00,0.00,0.00,0.00,0.00,0.00,5.62,0.00
25%,8.0,2.38,2.27,5.44,2.38,2.50,2.56,1.68,1.80,1.64,2.12,1.70,1.52,37.04,54.90
50%,8.0,3.09,2.81,8.06,3.00,2.97,3.22,2.25,2.41,2.13,2.48,2.19,2.19,60.00,65.97
75%,8.0,3.50,3.08,8.77,3.45,3.15,3.56,2.48,2.69,2.39,2.98,2.56,2.50,85.89,84.51
max,8.0,4.00,4.00,10.00,4.00,4.00,4.00,3.44,3.60,3.25,4.00,3.62,2.94,188.57,100.00


In [46]:
participant_features.describe().round(2)

,Number_of_sessions,Mean_engagement,Mean_emotional_connection,Mean_verbal_participation,Mean_attention,Mean_enjoyment,Mean_story_understanding,Mean_learning_application,Mean_confidence,Mean_generalisation,Mean_story_recall,Mean_reflection,Mean_real_life_connection,Mean_response_time_seconds,Mean_success_percentage
count,32.0,32.00,32.00,32.00,32.00,32.00,32.00,32.00,32.00,32.00,32.00,32.00,32.00,27.00,31.00
mean,8.0,2.86,2.62,7.03,2.82,2.77,2.91,2.08,2.18,1.87,2.34,1.94,1.89,69.82,67.91
std,0.0,0.83,0.79,2.37,0.81,0.78,0.86,0.84,0.91,0.87,1.02,0.97,0.87,48.67,22.89
min,8.0,0.40,0.10,0.40,0.30,0.30,0.00,0.00,0.00,0.00,0.00,0.00,0.00,5.62,0.00
25%,8.0,2.38,2.27,5.44,2.38,2.50,2.56,1.68,1.80,1.64,2.12,1.70,1.52,37.04,54.90
50%,8.0,3.09,2.81,8.06,3.00,2.97,3.22,2.25,2.41,2.13,2.48,2.19,2.19,60.00,65.97
75%,8.0,3.50,3.08,8.77,3.45,3.15,3.56,2.48,2.69,2.39,2.98,2.56,2.50,85.89,84.51
max,8.0,4.00,4.00,10.00,4.00,4.00,4.00,3.44,3.60,3.25,4.00,3.62,2.94,188.57,100.00


In [47]:
participant_features.to_csv(
    processed_data_folder / "participant_features.csv",
    index=True
)

In [48]:
list(processed_data_folder.glob("*"))

[WindowsPath('c:/Users/puspi/OneDrive/Desktop/Dissertation/data/processed/cleaned_data.csv'),
 WindowsPath('c:/Users/puspi/OneDrive/Desktop/Dissertation/data/processed/participant_features.csv')]